Učitati skup podataka 20newsgroups.

In [1]:
import numpy as np
import pandas as pd

data = pd.read_csv('20newsgroups.csv')

data.head()

,text,target
0,From: lerxst@wam.umd.edu (where's my thing)\nS...,7
1,From: guykuo@carson.u.washington.edu (Guy Kuo)...,4
2,From: twillis@ec.ecn.purdue.edu (Thomas E Will...,4
3,From: jgreen@amber (Joe Green)\nSubject: Re: W...,1
4,From: jcm@head-cfa.harvard.edu (Jonathan McDow...,14


• Ograničiti broj instanci na 200, vodeći računa o raspodeli klasa. 

In [2]:
from sklearn.model_selection import train_test_split

data_200, _ = train_test_split(data, train_size=200, stratify=data['target'], random_state=67)

• Koliko ima različitih klasa? Ako je potrebno, podeliti skup podataka na deo za obučavanje i 
deo za testiranje modela.

In [3]:
num = len(np.unique(data['target']))
print(f'Postoji {num} razlicitih klasa')

# ne treba podela na trening i test jer radimo klasterovanje

Postoji 20 razlicitih klasa


• Pretvoriti tekstualne podatke u TF-IDF matricu. 

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tv = TfidfVectorizer()
X_tv = tv.fit_transform(data_200['text'])

• Da li je potrebno dodatno pretprocesiranje podataka? Objasniti odgovor i ako je potrebno 
primeniti metode pretprocesiranja.

In [5]:
data_200['text']

6164     From: john@gu.uwa.edu.au (John West)\nSubject:...
11110    From: chudel@watarts.uwaterloo.ca (Chris Hudel...
10448    From: sandvik@newton.apple.com (Kent Sandvik)\...
1081     From: mveraart@fel.tno.nl (Mario Veraart)\nSub...
1534     From: I3150101@dbstu1.rz.tu-bs.de (Benedikt Ro...
                               ...                        
6806     From: brad@clarinet.com (Brad Templeton)\nSubj...
2083     From: brain@cbnewsj.cb.att.com (harish.s.mangr...
6241     From: golchowy@alchemy.chem.utoronto.ca (Geral...
8109     From: zeno@ccwf.cc.utexas.edu (S. Hsieh)\nSubj...
11031    From: keiths@spider.co.uk (Keith Smith)\nSubje...
Name: text, Length: 200, dtype: object

• Primeniti K-means algoritam za klasterovanje sa 20 klastera.

In [6]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=20, random_state=67)
kmeans_labels = kmeans.fit_predict(X_tv)

• Primeniti hijerarhijsko klasterovanje sa 20 klastera. 

In [7]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=100)
X_reduced = svd.fit_transform(X_tv)

from sklearn.cluster import AgglomerativeClustering

agc = AgglomerativeClustering(n_clusters=20)
agc_labels = agc.fit_predict(X_reduced)

• Ispisati silueta koeficijente za pronađene klastere. Da li su njihove vrednosti dobre? Koja 
metoda klasterovanja nam daje bolju vrednost? Kolika bi bila idealna vrednost silueta 
koeficijenta? Da li je silueta koeficijent adekvatna metrika za ovaj skup? Ako ne, zbog čega?

In [8]:
from sklearn.metrics import silhouette_score

sil_kmeans = silhouette_score(X_reduced, kmeans_labels)
sil_agc = silhouette_score(X_reduced, agc_labels)

sil_kmeans, sil_agc

(0.02205005772869276, 0.022375941983087132)

• Primeniti PCA proceduru sa dve komponente nad ulaznim podacima. Koliki procenat 
varijanse se ovime održao? Nacrtati grafik transformisanog skupa u dve dimenzije i instance
obojiti na osnovu toga kom klasteru pripadaju (jedan grafik za K-means, a jedan grafik za 
hijerarhijsko klasterovanje). 

In [9]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_reduced)

• Uporediti klastere dobijene pomoću ove dve metode sa originalnim klasama datim u skupu 
podataka. Koliko se dobro podudaraju?

In [10]:
from sklearn.metrics import adjusted_rand_score

ari_kmeans = adjusted_rand_score(data_200['target'], kmeans_labels)
ari_agc = adjusted_rand_score(data_200['target'], agc_labels)

ari_kmeans, ari_agc

(0.02290296873639738, 0.049922065087510244)